# 10 – Profiles

Esplorazione e data cleaning del dataset `profiles.csv`.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL (chiave primaria) |
| `gender` | Genere dichiarato (`Male`, `Female`, `Non-Binary`) |
| `birthday` | Data di nascita (testo libero) |
| `location` | Paese dell'utente (22 valori categorici) |
| `joined` | Data di iscrizione a MAL |
| `watching` | Numero di anime attualmente in visione |
| `completed` | Numero di anime completati |
| `on_hold` | Numero di anime in pausa |
| `dropped` | Numero di anime abbandonati |
| `plan_to_watch` | Numero di anime in lista d'attesa |

## 1. Import e caricamento dati

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

In [2]:
df_pr = pd.read_csv('../datasets/profiles.csv')
print(f'Shape: {df_pr.shape}')
df_pr.info()
df_pr.head()

Shape: (337155, 10)
<class 'pandas.DataFrame'>
RangeIndex: 337155 entries, 0 to 337154
Data columns (total 10 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   username       337154 non-null  str  
 1   gender         166279 non-null  str  
 2   birthday       121329 non-null  str  
 3   location       337155 non-null  str  
 4   joined         335479 non-null  str  
 5   watching       335477 non-null  str  
 6   completed      335477 non-null  str  
 7   on_hold        335477 non-null  str  
 8   dropped        335477 non-null  str  
 9   plan_to_watch  335477 non-null  str  
dtypes: str(10)
memory usage: 25.7 MB


,username,gender,birthday,location,joined,watching,completed,on_hold,dropped,plan_to_watch
0,ishikawas,NaN,NaN,South Korea,NaN,NaN,NaN,NaN,NaN,NaN
1,CKK2,NaN,NaN,United States,"Dec 1, 2018",3,182,15,0,405
2,--------788,Female,NaN,Mexico,"Oct 4, 2022",1,64,0,0,1
3,potatoaris,NaN,NaN,Spain,"Oct 2, 2018",5,1,0,0,4
4,Rinrintan,NaN,NaN,Japan,"May 12, 2019",20,311,40,16,34


In [3]:
n_originale = len(df_pr)

## 1.1 Rimozione riga senza `username`

È presente 1 riga con `username` null. Senza identificatore il profilo è inutilizzabile → rimozione.

In [4]:
print('Riga con username null:')
print(df_pr[df_pr['username'].isna()])

df_pr.dropna(subset=['username'], inplace=True)
print(f'\nRighe dopo rimozione: {len(df_pr):,}')

Riga con username null:
       username gender birthday location        joined watching completed  \
271344      NaN    NaN      NaN   Mexico  Jul 28, 2007        2         2   

       on_hold dropped plan_to_watch  
271344       2       0             1  

Righe dopo rimozione: 337,154


## 1.2 Rimozione profili senza dati statistici

1.678 righe hanno tutte le colonne statistiche (`watching`, `completed`, `on_hold`, `dropped`, `plan_to_watch`) e la data di iscrizione nulle: si tratta di profili vuoti senza alcun dato utile → rimozione.

In [5]:
stat_cols = ['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']
mask_no_stats = df_pr[stat_cols].isna().all(axis=1)
print(f'Profili senza dati statistici : {mask_no_stats.sum():,}')
print(f'  di cui anche senza joined   : {(mask_no_stats & df_pr["joined"].isna()).sum():,}')
print()
print('Esempio profili vuoti:')
print(df_pr[mask_no_stats][['username', 'gender', 'location', 'joined'] + stat_cols].head())

df_pr = df_pr[~mask_no_stats].copy()
print(f'\nRighe prima della rimozione : {n_originale - 1:,}')
print(f'Righe dopo la rimozione     : {len(df_pr):,}')

Profili senza dati statistici : 1,678
  di cui anche senza joined   : 1,676

Esempio profili vuoti:
       username gender       location joined watching completed on_hold  \
0     ishikawas    NaN    South Korea    NaN      NaN       NaN     NaN   
12  potatobeann    NaN         France    NaN      NaN       NaN     NaN   
20   potatochib    NaN    South Korea    NaN      NaN       NaN     NaN   
36     ishixawa    NaN          Japan    NaN      NaN       NaN     NaN   
71   ---KING---    NaN  United States    NaN      NaN       NaN     NaN   

   dropped plan_to_watch  
0      NaN           NaN  
12     NaN           NaN  
20     NaN           NaN  
36     NaN           NaN  
71     NaN           NaN  

Righe prima della rimozione : 337,154
Righe dopo la rimozione     : 335,476


## 2. Analisi colonna per colonna

### 2.1 `username`

In [6]:
analyze(df_pr['username'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'username'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    username
  dtype                          str
  Shape                          335,476
  Index range                    1 … 337154
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     335,476
  Non-null values                335,476  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  335,476  (100.00%)
  Duplicate values               0

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     1  chars
  Max

**Nessuna pulizia necessaria.**  
Nessun null (rimosso al passo 1.1), nessun duplicato. È la chiave primaria del dataset.

In [7]:
print(f'Null in username      : {df_pr["username"].isna().sum()}')
print(f'Duplicati in username : {df_pr["username"].duplicated().sum()}')

Null in username      : 0
Duplicati in username : 0


### 2.2 `gender`

In [8]:
analyze(df_pr['gender'])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'gender'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    gender
  dtype                          str
  Shape                          335,476
  Index range                    1 … 337154
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     335,476
  Non-null values                166,277  [███████████████░░░░░░░░░░░░░░░]  49.6%
  Null / NaN                     169,199  (50.44%)
  Empty strings                  0
  Unique values                  3  (0.00%)
  Duplicate values               166,274

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     4  chars
  

**Nessuna pulizia necessaria.**  
I null (~170.876, ~51%) sono strutturali: la maggior parte degli utenti MAL non dichiara il genere. I tre valori (`Male`, `Female`, `Non-Binary`) sono già consistenti.

In [9]:
print(f'Null in gender : {df_pr["gender"].isna().sum():,} ({df_pr["gender"].isna().mean()*100:.1f}%)')
print()
print('Distribuzione gender:')
print(df_pr['gender'].value_counts())

Null in gender : 169,199 (50.4%)

Distribuzione gender:
gender
Male          120747
Female         41313
Non-Binary      4217
Name: count, dtype: int64


### 2.3 `birthday`

In [10]:
analyze(df_pr['birthday'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'birthday'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    birthday
  dtype                          str
  Shape                          335,476
  Index range                    1 … 337154
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     335,476
  Non-null values                121,327  [███████████░░░░░░░░░░░░░░░░░░░]  36.2%
  Null / NaN                     214,149  (63.83%)
  Empty strings                  0
  Unique values                  11,344  (3.38%)
  Duplicate values               109,983

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     1  c

**Pulizia necessaria:**  
- I valori sono stringhe in formato libero (es. `'Jul 7, 1989'`, `'2003'`, `'Jan 1'`) → conversione a `datetime64` con `errors='coerce'`
- Le date parziali (solo anno o solo giorno/mese) non sono parsabili come data completa → diventano `NaT`
- I ~215.826 null originali (~64%) sono strutturali: molti utenti non inseriscono la data di nascita su MAL

In [11]:
n_null_before = df_pr['birthday'].isna().sum()
n_nonnull_before = df_pr['birthday'].notna().sum()

df_pr['birthday'] = pd.to_datetime(df_pr['birthday'], errors='coerce')

n_null_after = df_pr['birthday'].isna().sum()
n_coerced = n_null_after - n_null_before

print(f'Valori non-null prima della conversione : {n_nonnull_before:,}')
print(f'  → convertiti correttamente            : {n_nonnull_before - n_coerced:,}')
print(f'  → coercizzati a NaT (date parziali)   : {n_coerced:,}')
print(f'Null totali dopo la conversione         : {n_null_after:,} ({n_null_after/len(df_pr)*100:.1f}%)')
print(f'birthday dtype : {df_pr["birthday"].dtype}')
print(f'Range          : {df_pr["birthday"].min()} → {df_pr["birthday"].max()}')

Valori non-null prima della conversione : 121,327
  → convertiti correttamente            : 86,055
  → coercizzati a NaT (date parziali)   : 35,272
Null totali dopo la conversione         : 249,421 (74.3%)
birthday dtype : datetime64[us]
Range          : 0001-01-01 00:00:00 → 9999-09-03 00:00:00


### 2.4 `location`

In [12]:
analyze(df_pr['location'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'location'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    location
  dtype                          str
  Shape                          335,476
  Index range                    1 … 337154
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     335,476
  Non-null values                335,476  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  22  (0.01%)
  Duplicate values               335,454

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     5  chars
  Max 

**Nessuna pulizia necessaria.**  
Nessun null. La colonna presenta esattamente 22 valori distinti (paesi) uniformemente distribuiti, indice che si tratta di un campo categorico a selezione fissa (dropdown) anziché testo libero. Il Giappone è il paese più rappresentato, coerente con la base utenti di MAL.

In [13]:
print(f'Null in location  : {df_pr["location"].isna().sum()}')
print(f'Valori unici      : {df_pr["location"].nunique()}')
print()
print('Distribuzione location:')
print(df_pr['location'].value_counts())

Null in location  : 0
Valori unici      : 22

Distribuzione location:
location
Japan             97820
United States     64930
Germany           26226
United Kingdom    19433
Thailand          16332
Argentina         13165
China             13128
Spain             12837
France             9983
Australia          9651
Mexico             6597
South Korea        6490
Turkey             6485
Italy              6440
Indonesia          3314
Brazil             3279
Vietnam            3257
South Africa       3253
Philippines        3240
Egypt              3222
India              3207
Canada             3187
Name: count, dtype: int64


### 2.5 `joined`

In [14]:
analyze(df_pr['joined'])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'joined'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    joined
  dtype                          str
  Shape                          335,476
  Index range                    1 … 337154
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     335,476
  Non-null values                335,476  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  6,802  (2.03%)
  Duplicate values               328,674

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     11  chars
  Ma

**Pulizia necessaria:**  
I valori sono stringhe in formato `'Dec 1, 2018'` → conversione a `datetime64`.  
I null residui (~2, dopo la rimozione dei profili vuoti al passo 1.2) sono strutturali.

In [15]:
df_pr['joined'] = pd.to_datetime(df_pr['joined'], errors='coerce')
print(f'joined dtype  : {df_pr["joined"].dtype}')
print(f'Null residui  : {df_pr["joined"].isna().sum()}')
print(f'Range         : {df_pr["joined"].min()} → {df_pr["joined"].max()}')

joined dtype  : datetime64[us]
Null residui  : 0
Range         : 2004-11-05 00:00:00 → 2025-11-02 00:00:00


### 2.6 Colonne statistiche: `watching`, `completed`, `on_hold`, `dropped`, `plan_to_watch`

Le cinque colonne sono caricate come `str` e alcune contengono numeri con separatore delle migliaia (es. `'1,008'`). Pulizia uniforme: rimozione del separatore e conversione a `int64`.

In [16]:
stat_cols = ['watching', 'completed', 'on_hold', 'dropped', 'plan_to_watch']

for col in stat_cols:
    analyze(df_pr[col])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'watching'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    watching
  dtype                          str
  Shape                          335,476
  Index range                    1 … 337154
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     335,476
  Non-null values                335,476  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  942  (0.28%)
  Duplicate values               334,534

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                     1  chars
  Max

**Pulizia necessaria per tutte e cinque le colonne:**  
- Alcune celle contengono numeri con separatore migliaia (virgola), es. `'1,008'` → rimozione della virgola
- Dtype `str` → conversione a `int64`
- Dopo la rimozione dei profili vuoti (passo 1.2) non ci sono null residui

In [17]:
for col in stat_cols:
    # Rimuovi separatore delle migliaia
    df_pr[col] = df_pr[col].str.replace(',', '', regex=False)
    # Converti a int64
    df_pr[col] = pd.to_numeric(df_pr[col], errors='coerce').astype('Int64')

print('Dtypes colonne statistiche dopo la conversione:')
print(df_pr[stat_cols].dtypes)
print()
print('Null residui:')
print(df_pr[stat_cols].isnull().sum())
print()
print('Range valori:')
print(df_pr[stat_cols].agg(['min', 'max']))

Dtypes colonne statistiche dopo la conversione:
watching         Int64
completed        Int64
on_hold          Int64
dropped          Int64
plan_to_watch    Int64
dtype: object

Null residui:
watching         0
completed        0
on_hold          0
dropped          0
plan_to_watch    0
dtype: int64

Range valori:
     watching  completed  on_hold  dropped  plan_to_watch
min         0          0        0        0              0
max     27020      28313    18403    27277          27193


### 2.7 Chiave primaria `username`

Verifichiamo che `username` sia univoco dopo tutte le operazioni di pulizia.

In [18]:
n_pk_dup = df_pr.duplicated(subset=['username'], keep=False).sum()
print(f'Duplicati su username: {n_pk_dup}')

if n_pk_dup > 0:
    print('→ Rimozione duplicati sulla chiave primaria...')
    df_pr.drop_duplicates(subset=['username'], keep='first', inplace=True)
    print(f'Righe dopo la rimozione: {len(df_pr):,}')
else:
    print('→ Chiave primaria già univoca, nessuna operazione richiesta.')

Duplicati su username: 0
→ Chiave primaria già univoca, nessuna operazione richiesta.


## 3. Data Cleaning

In [19]:
print('=== Riepilogo Dataset Pulito ===')
print(f'Righe originali      : {n_originale:>10,}')
print(f'Righe dopo cleaning  : {len(df_pr):>10,}')
print(f'Righe rimosse totali : {n_originale - len(df_pr):>10,}')
print()
print('Dtypes finali:')
print(df_pr.dtypes)
print()
print('Valori mancanti residui:')
print(df_pr.isnull().sum())

=== Riepilogo Dataset Pulito ===
Righe originali      :    337,155
Righe dopo cleaning  :    335,476
Righe rimosse totali :      1,679

Dtypes finali:
username                    str
gender                      str
birthday         datetime64[us]
location                    str
joined           datetime64[us]
watching                  Int64
completed                 Int64
on_hold                   Int64
dropped                   Int64
plan_to_watch             Int64
dtype: object

Valori mancanti residui:
username              0
gender           169199
birthday         249421
location              0
joined                0
watching              0
completed             0
on_hold               0
dropped               0
plan_to_watch         0
dtype: int64


In [20]:
df_pr.head()

,username,gender,birthday,location,joined,watching,completed,on_hold,dropped,plan_to_watch
1,CKK2,NaN,NaT,United States,2018-12-01,3,182,15,0,405
2,--------788,Female,NaT,Mexico,2022-10-04,1,64,0,0,1
3,potatoaris,NaN,NaT,Spain,2018-10-02,5,1,0,0,4
4,Rinrintan,NaN,NaT,Japan,2019-05-12,20,311,40,16,34
5,Karincakes,NaN,NaT,Turkey,2015-01-05,79,2899,22,17,508


## 4. Salvataggio dataset pulito

In [21]:
df_pr.to_csv('../datasets_cleaned/profiles_clean.csv', index=False)
print('Salvato: datasets_cleaned/profiles_clean.csv')

Salvato: datasets_cleaned/profiles_clean.csv
